# Mass Spectrometry Analysis of Extracellular Vesicles Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library, leveraging Croissant schema semantics and referencing entities by their `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL pointing to the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" record package.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {getattr(metadata, 'name')}")
print(getattr(metadata, 'description'))

## 2. Data Overview
Review available record sets, fields, and their IDs (using their `@id`).

In [ ]:
# List record sets by @id
print("Record sets available (by @id):")
record_sets = getattr(metadata, 'record_sets', None)
if record_sets is None:
    # Try newer metadata interface
    if hasattr(metadata, 'record_set'):
        record_sets = getattr(metadata, 'record_set')
    else:
        record_sets = []

if isinstance(record_sets, dict):
    record_sets = [record_sets]
elif not isinstance(record_sets, list):
    record_sets = list(record_sets) if record_sets else []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or (rs['@id'] if isinstance(rs, dict) and '@id' in rs else None)
    rs_name = getattr(rs, 'name', None) or (rs['name'] if isinstance(rs, dict) and 'name' in rs else None)
    print(f"- @id: {rs_id}, name: {rs_name}")

if not record_sets:
    print("(No record sets found in metadata; the dataset interface will still attempt to enumerate via mlcroissant)")

# Explore field IDs for the main record set (using mlcroissant helper)
print("\nEnumerating all record sets in the Croissant schema using dataset.list_record_sets():\n")
avail_record_set_ids = dataset.list_record_sets()
for rs_id in avail_record_set_ids:
    print(f"Record set @id: {rs_id}")
    fields = dataset.list_fields(record_set=rs_id)
    print(f"  Field @ids:")
    for field_id in fields:
        print(f"    - {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s are taken from the previous overview.

**Note:** For this dataset, we dynamically extract the available record set(s) and their field columns using only their `@id` values.

In [ ]:
# Extract data from all detected record sets
avail_record_set_ids = dataset.list_record_sets()
dataframes = {}

# Load the data into pandas DataFrames
for record_set_id in avail_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for record set @id: {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set @id '{record_set_id}':")
    print(df.columns.tolist())
    print(df.head())

# For simplicity, select the first non-empty record set for further analysis
main_record_set_id = next(iter(dataframes.keys())) if len(dataframes) > 0 else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates numeric field normalization, filtering, and grouping using only `@id` field references.

*If you want to analyze a different record set, set `main_record_set_id` to the appropriate `@id` from the previous step.*

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"\nEDA for record set @id: {main_record_set_id}")
    # Guess numeric fields by dtype
    numeric_columns = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if len(numeric_columns) == 0:
        print("No numeric fields found for EDA. Please inspect the DataFrame columns above.")
        numeric_field_id = None
    else:
        numeric_field_id = numeric_columns[0]  # Just use the first available numeric field by @id
        print(f"Using numeric field @id: {numeric_field_id}")

    # Filter for values above a threshold (e.g., 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (string/object dtype not the numeric field itself)
    object_columns = [col for col in df.columns if (df[col].dtype == 'object' and col != numeric_field_id)]
    group_field_id = object_columns[0] if len(object_columns) > 0 else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"\nGrouped (mean) by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("\nNo suitable categorical field detected for grouping.")
else:
    print("No available record set for EDA. Please check previous steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below we plot a histogram of the numeric field if available, referencing columns by their full `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to:
 - Load Croissant metadata directly from the FAIR² schema URL.
 - Enumerate all record sets and their fields by `@id`.
 - Dynamically extract tabular data from the dataset, referencing columns by their `@id`.
 - Perform basic EDA (filtering, normalization, grouping) on numeric fields and visualize their distributions.

This ensures robust and reproducible analytics referencing dataset structure according to the FAIR and Croissant standards.